# `odi_to_databricks_conversion.txt`

**Conversion Timestamp:** 2024-07-30 12:00:00 UTC

**Description:** This notebook performs a full load insert into the `trg_dep` table from the `departments` table in the `HR` schema.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "", "ODI Session Number")

# ETL Parameters

Displaying the values for the ETL parameters defined as widgets.

In [ ]:
display(spark.sql("""
  SELECT
    '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
    '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
    '${ETL_PROC_WID}' AS ETL_PROC_WID,
    '${ODI_SESS_NO}' AS ODI_SESS_NO
"""))

# Target Table Load

This section handles the loading of the `trg_dep` table.

In [ ]:
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}
%sql
INSERT INTO workspace.hr.trg_dep
  (
    department_id,
    department_name,
    manager_id,
    location_id
  )
SELECT
  departments.department_id,
  departments.department_name,
  departments.manager_id,
  departments.location_id
FROM
  workspace.hr.departments AS departments;

# Cleanup

There are no temporary tables to clean up for this specific process.

# Validation

Confirming the record count in the target table after the load.

In [ ]:
%sql
SELECT
  COUNT(*) AS total_records
FROM
  workspace.hr.trg_dep;

# Conversion Notes and Manual Actions Required

1.  **Schema and Table Naming**: Oracle schema `HR` and table names `TRG_DEP`, `DEPARTMENTS` have been converted to lowercase and prefixed with `workspace.`: `workspace.hr.trg_dep`, `workspace.hr.departments`.
2.  **Oracle Hints Removed**: The Oracle hint `/*+ APPEND PARALLEL */` has been removed as it is not applicable to Databricks Delta Lake and Spark SQL.
3.  **No ODI Variables**: The original SQL did not contain any ODI variables (`#GLOBAL`, `#SCHEMA`), so no widget-to-SQL parameter mapping was required in the `INSERT` statement.
4.  **No Type Conversions**: The provided SQL only involved `INSERT ... SELECT` and did not include any DDL, so explicit data type conversions (e.g., `NUMBER(p,0)` to `BIGINT`) were not directly applied in this specific SQL block, but assumed to be handled at table creation time for `TRG_DEP`.
5.  **No `OPTIMIZE ZORDER BY`**: This simple INSERT statement does not trigger `OPTIMIZE ZORDER BY` as it is typically applied after MERGE operations or large data changes to a target table. If `trg_dep` requires ZORDERing for query performance, it should be added as a separate step.
6.  **No Staging/Flow Tables**: The original ODI logic was a direct insert and did not involve `C$` or `I$` staging/flow tables, hence no logic for creating, populating, or dropping such tables is included.